# Notebook 3. The structure priced in bits (Fig. 3)

Reproduces Fig. 3 and SI Tables S4 and S5. Reads the folders `data/paper2` and `data/fig3_data`. Run time under a minute.

In [1]:
import contextlib
import glob
import io
import os
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx
from scipy.stats import spearmanr

# the folder with the data of the second paper and the folder of the Fig. 3 data, in the folder data of the
# repository, or the folders named by the environment variables P2_DIR and F3_DIR
P2_DIR = os.environ.get('P2_DIR') or '../data/paper2'
F3_DIR = os.environ.get('F3_DIR') or '../data/fig3_data'

TOPOS = ['LAT', 'WS', 'ER', 'BA']                 # from regular to scale-free, the order of Fig. 2
KIND = {'LAT': 'regular', 'WS': 'small-world', 'ER': 'random', 'BA': 'scalefree'}   # the names of paper 2

# values published in the second paper: Table 1 of the main text and Supplementary Table 1
PUB = dict(H={'LAT': 6.6439, 'WS': 6.5832, 'ER': 6.4820, 'BA': 6.3176},
           dH={'LAT': 0.000, 'WS': 0.061, 'ER': 0.162, 'BA': 0.326},
           deff={'LAT': 100.0, 'WS': 95.9, 'ER': 89.4, 'BA': 79.8},
           kappa={'LAT': 6.00, 'WS': 6.09, 'ER': 6.90, 'BA': 10.10},
           H_deg={'LAT': 6.6439, 'WS': 6.6324, 'ER': 6.5313, 'BA': 6.2865})
print('data of the second paper in', P2_DIR, '| Fig. 3 data in', F3_DIR)

data of the second paper in ../data/paper2 | Fig. 3 data in ../data/fig3_data


## Step 0. The code of the second paper

In [2]:
# fig3_publication_4, cell 2: the core, Methods Eqs. 1 to 13 of the second paper, code unchanged.
SRC_CORE = r'''# ============================================================================
# CORE  -  Equations 1 to 13 of the second paper (Methods numbering).
# The same cell in every notebook of the second paper.
# ============================================================================
# Eq 1   p_i = c_i / sum_j c_j                       coupling share
# Eq 2   H = - sum_i p_i log2 p_i                     control entropy (avg information)
# Eq 3   p_i = C_i / sum_j C_j,  A C = lambda C       eigenvector weighting
# Eq 4   J = H / log2 N                               evenness
# Eq 5   D_eff = 2^H                                  effective count of DOF
# Eq 6   D_eff / N = 2^(-dH),  dH = log2 N - H        retained fraction / compression
# Eq 7   H <= log2 N                                  the bound (Gibbs / Jensen)
# Eq 8   dH/deps = log2(p_j / p_i)                    transfer gradient, concentration lowers H
# Eq 9   H = H2(phi) + phi log2 h + (1-phi) log2(N-h) leader-follower closed form
# Eq 10  kappa = <k^2> / <k>                          degree heterogeneity
# Eq 11  r                                            degree assortativity
# Eq 12  dh = log2(sigma_A / sigma_B)                 outcome entropy difference
# Eq 13 (Hebbian) lives in s4, since it needs the parent model's activations.
#
# I(p) = log2(1/p) is the summand of Eq 2 (Introduction Eq 2).
# Compression dH = log2 N - H is the bits saved (Introduction Eq 5), used by Eq 6.
# ============================================================================
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

CORE_VERSION = "v5 / 24 July 2026 (Methods Eq 1-13)"

def share(c):
    """Eq 1. Normalise any non-negative vector into coupling shares."""
    c = np.asarray(c, dtype=float).ravel()
    if np.any(c < 0):
        raise ValueError("coupling values must be non-negative")
    t = c.sum()
    if t <= 0:
        raise ValueError("total coupling is zero")
    return c / t

def information(p):
    """Information of a share, in bits, the summand of Eq 2. Small share -> large information."""
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.inf)
    np.log2(1.0 / p, out=out, where=(p > 0))
    return out

def shannon_entropy(c):
    """Eq 2. H as the average of the information over the distribution. Input may be raw."""
    p = share(c)
    p = p[p > 0]
    return float((p * information(p)).sum())

def max_entropy(n):
    """Eq 7 (right side). The ceiling log2 N, attained only by the uniform distribution."""
    if n < 1:
        raise ValueError("n must be at least 1")
    return float(np.log2(n))

def evenness(h, n):
    """Eq 4. J = H / log2 N. Compare systems of different size with this."""
    m = max_entropy(n)
    return 1.0 if m == 0 else float(h / m)

def effective_dof(h):
    """Eq 5. D_eff = 2^H, a number of equally coupled degrees of freedom."""
    return float(2.0 ** h)

def compression(H, N):
    """dH = log2 N - H (Introduction Eq 5). Bits saved against even coupling.
    Regular gives 0 exactly; the scale-free ensemble gives 0.3130 at N = 100.
    """
    return float(np.log2(N) - H)

def retained_fraction(H, N):
    """Eq 6. 2^(-dH) = D_eff / N, the fraction of degrees of freedom still to be
    specified. Equals 2^H / N. Scale-free gives 0.8050, the 80.5 out of 100.
    """
    return float(2.0 ** (-(np.log2(N) - H)))

def coupling_shares(G, weight="degree"):
    """Eq 1 with c = degree, or Eq 3 with c = eigenvector centrality."""
    nodes = list(G.nodes())
    if weight == "degree":
        v = np.array([G.degree(n) for n in nodes], dtype=float)
    elif weight == "eigenvector":
        c = nx.eigenvector_centrality_numpy(G)
        v = np.abs(np.array([c[n] for n in nodes], dtype=float))
    else:
        raise ValueError("weight must be 'degree' or 'eigenvector'")
    return share(v)

def control_entropy(G, weight="degree"):
    """Eq 2 applied to Eq 1 or Eq 3."""
    return shannon_entropy(coupling_shares(G, weight=weight))

def strength_shares(W):
    """Eq 1 for a weighted structure: c_i = node strength = sum_j W_ij."""
    W = np.asarray(W, dtype=float)
    if W.ndim != 2 or W.shape[0] != W.shape[1]:
        raise ValueError("W must be square")
    return share(W.sum(axis=1))

def transfer_gradient(p, i, j):
    """Eq 8. dH/deps = log2(p_j / p_i) when eps moves coupling FROM j TO i.
    Negative whenever p_i > p_j, so any transfer toward an already-strong
    element strictly lowers H. i is the element that receives coupling.
    """
    p = np.asarray(p, dtype=float)
    return float(np.log2(p[j] / p[i]))

def _binary_entropy(phi):
    """H2(phi) = -phi log2 phi - (1-phi) log2(1-phi), with the 0 log 0 = 0 rule."""
    phi = float(phi)
    if phi <= 0.0 or phi >= 1.0:
        return 0.0
    return float(-phi * np.log2(phi) - (1.0 - phi) * np.log2(1.0 - phi))

def two_level_entropy(N, h, phi):
    """Eq 9. H = H2(phi) + phi log2 h + (1-phi) log2(N-h).
    h leaders share phi equally and N-h followers share (1-phi) equally.
    Returns log2 N exactly when phi = h/N, the built-in unit test.
    """
    if not (1 <= h < N):
        raise ValueError("need 1 <= h < N")
    if not (0.0 <= phi <= 1.0):
        raise ValueError("phi must be in [0, 1]")
    lead = phi * np.log2(h) if h > 0 else 0.0
    return float(_binary_entropy(phi) + lead + (1.0 - phi) * np.log2(N - h))

def degree_heterogeneity(G):
    """Eq 10. kappa = <k^2> / <k>, the parent paper's normalisation.

    NOTE ON NORMALISATION. The handwritten note of 23 July writes the ratio as
    <k^2>/<k>^2. That differs from this by a factor <k>, and the parent paper's
    published values (6.84, 6.09, 9.67) are on THIS definition. Table 1 of the
    manuscript uses this one so the two papers stay comparable. Convert with
    kappa_alt = kappa / <k> if a source uses the other.
    """
    k = np.array([d for _, d in G.degree()], dtype=float)
    if k.mean() <= 0:
        return float("nan")
    return float((k ** 2).mean() / k.mean())

def degree_assortativity(G):
    """Eq 11. Pearson correlation of degrees at the two ends of an edge.

    Undefined for a regular graph, where the degrees carry no variance. Returns
    nan there rather than a fabricated zero.
    """
    k = np.array([d for _, d in G.degree()], dtype=float)
    if k.std() == 0 or G.number_of_edges() == 0:
        return float("nan")
    try:
        r = nx.degree_assortativity_coefficient(G)
    except (ValueError, ZeroDivisionError):
        return float("nan")
    return float(r) if np.isfinite(r) else float("nan")

def gini(x):
    """Gini coefficient, matching the parent paper's definition."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2.0 * (idx * x).sum()) / (n * x.sum()) - (n + 1.0) / n)

GAUSS_CONST = 0.5 * np.log2(2.0 * np.pi * np.e)

def outcome_entropy_gaussian(sigma):
    """h = 0.5 log2(2 pi e sigma^2). A DIFFERENTIAL entropy, not Eq 2.

    It shares the unit and the reading of predictability with control entropy
    and is a different quantity on a different object. Only differences on a
    common scale are interpreted, which is why Eq 12 exists.
    """
    s = np.asarray(sigma, dtype=float)
    if np.any(s <= 0):
        raise ValueError("sigma must be positive")
    return GAUSS_CONST + np.log2(s)

def delta_outcome_entropy(sigma_a, sigma_b):
    """Eq 12. The additive constant and the unit both cancel."""
    return float(np.log2(float(sigma_a) / float(sigma_b)))

def predictability_ratio(dh):
    """R = 2^dh, the predictability ratio that follows Eq 12."""
    return float(2.0 ** float(dh))

print("core loaded:", CORE_VERSION)
'''

In [3]:
# fig3_publication_4, cell 6: the four matched ensembles, code unchanged.
SRC_ENS = r'''# =========================================================================
# THE FOUR MATCHED ENSEMBLES
# =========================================================================
# Method as published: kappa and the assortativity are read on the full
# generated graph, while the eigenvector calculation uses the largest
# connected component, because the leading eigenvector is not uniquely
# defined on a disconnected graph.
#
# The regular ensemble is a ring lattice. The earlier code used
# random_regular_graph, which is not a lattice. Every reported number is
# identical either way, since both give every element exactly six couplings,
# so kappa is 6.00 and H is log2 N to machine precision. Only the picture
# changes, and only the picture was wrong.
#
# The four ensembles are matched on the coupling budget rather than near it.
# Every realisation of every ensemble holds exactly 300 links among 100
# elements, so the mean degree is 6.0000 in all four and the only thing that
# differs is how those 300 links are distributed. Two generators had to change
# for that. The random ensemble is drawn with a fixed edge count instead of a
# fixed edge probability, which is the G(n,m) form of the same model rather
# than the G(n,p) form. The scale-free ensemble grows by preferential
# attachment from a twelve element ring lattice of the same mean degree, which
# is what brings its edge count to 300 as well. The plain m = 3 growth from a
# bare seed stops at 291 links and a mean degree of 5.82.
N, K_MEAN, N_REAL, SEED0 = 100, 6, 100, 42
M_LINKS = N * K_MEAN // 2                # 300 links in every realisation
BA_M, BA_N0 = 3, 12                      # 36 + 3 x 88 = 300


def make(kind, seed):
    if kind == 'regular':     return nx.watts_strogatz_graph(N, K_MEAN, 0.0, seed=seed)
    if kind == 'small-world': return nx.watts_strogatz_graph(N, K_MEAN, 0.1, seed=seed)
    if kind == 'random':      return nx.gnm_random_graph(N, M_LINKS, seed=seed)
    if kind == 'scalefree':
        g0 = nx.watts_strogatz_graph(BA_N0, K_MEAN, 0.0, seed=seed)
        return nx.barabasi_albert_graph(N, BA_M, seed=seed, initial_graph=g0)
    raise ValueError(kind)


def giant(G):
    return G if nx.is_connected(G) else G.subgraph(
        max(nx.connected_components(G), key=len)).copy()


KINDS = ['regular', 'small-world', 'random', 'scalefree']
LETTER = dict(zip(KINDS, 'ABCD'))
LABEL = {'regular': 'Regular lattice', 'small-world': 'Small-world',
         'random': 'Random', 'scalefree': 'Scale-free'}
GEN = {'regular': r'ring,  $\beta$ = 0', 'small-world': r'rewired,  $\beta$ = 0.1',
       'random': r'$G(n,m)$,  300 links', 'scalefree': r'preferential,  3 per element'}
ens = {}
for kind in KINDS:
    H_eig, H_deg, kap, rr, mk, degs, prof, nlk = [], [], [], [], [], [], [], []
    for s in range(SEED0, SEED0 + N_REAL):
        Gf = make(kind, s)                       # full graph
        G = giant(Gf)                            # largest component
        w = coupling_shares(G, 'eigenvector')
        H_eig.append(shannon_entropy(w))
        H_deg.append(control_entropy(G, 'degree'))
        kap.append(degree_heterogeneity(Gf))
        rr.append(degree_assortativity(Gf))
        d = np.array([d for _, d in Gf.degree()], dtype=float)
        mk.append(d.mean()); degs.append(d); nlk.append(Gf.number_of_edges())
        srt = np.sort(w)[::-1]                   # the ranked share profile
        if srt.size != N:                        # a realisation that lost a piece
            srt = np.interp(np.linspace(0.0, 1.0, N),
                            np.linspace(0.0, 1.0, srt.size), srt)
            srt = srt / srt.sum()                # keep the profile a distribution
        prof.append(srt)
    rr = np.array(rr, dtype=float); fin = rr[np.isfinite(rr)]
    ens[kind] = {'H': np.array(H_eig), 'Hdeg': np.array(H_deg),
                 'kappa': np.array(kap), 'k': np.array(mk),
                 'r': fin if fin.size else np.array([np.nan]),
                 'deg': np.concatenate(degs), 'm': np.array(nlk),
                 'P': np.array(prof),
                 'prof': np.mean(np.array(prof), axis=0)}

CEIL = max_entropy(N)                            # Eq 5


def dH(H):
    """Compression for display. The lattice sits at the ceiling by theorem, so
    never print its value as a negative zero."""
    v = compression(H, N)
    return 0.0 if abs(v) < 1e-9 else v


# the bootstrap the intervals are drawn from, computed here and not quoted
BOOT = 10000
rng = np.random.default_rng(0)
CI = {}
for kind in KINDS:
    v = ens[kind]['H']
    bs = np.array([rng.choice(v, size=len(v), replace=True).mean()
                   for _ in range(BOOT)])
    CI[kind] = tuple(np.percentile(bs, [2.5, 97.5]))

# one realisation of each, the one each panel draws
SHOW = {k: giant(make(k, SEED0)) for k in KINDS}
def drawn_shares(G):
    """The shares the picture is built from, rounded once at the source.

    nx.eigenvector_centrality_numpy returns values that differ in the last
    digits between runs. Layout already breaks its ties on rounded shares, and
    now that node colour and node area are continuous functions of the share
    the same discipline has to apply to them, or the same script renders two
    images that differ by a level or two of grey. Rounding at 12 decimals
    leaves nine significant figures on the smallest share in the figure and
    changes no reported number, because every value in Table 1 is computed
    from the unrounded ensemble arrays above."""
    return np.round(coupling_shares(G, 'eigenvector'), 12)


SHOW_W = {k: drawn_shares(SHOW[k]) for k in KINDS}
_WMAX = max(float(w.max()) for w in SHOW_W.values())
_PMAX = max(float(ens[k]['prof'].max()) for k in KINDS)


def nsize(G):
    """Node area for the 100 element structure, drawn inside the share plot, so the
    scale is small. Area is set by the number of links the element holds, on
    one scale across all four, so an element of six links is drawn at one size
    wherever it appears."""
    d = np.array([G.degree(n) for n in G.nodes()], dtype=float)
    return 0.8 + 36.0 * (d / _KMAX)


def spread(G, w):
    """One spring rule for all four ensembles, so every difference between
    the four pictures comes from the graphs and not from the drawing.

    The leading element is seeded at the centre and the rest on a spiral by
    rank in share, then the spring is run from there. Ties are broken on
    rounded shares, then degree, then label, because the eigenvector solver
    returns values that differ in the last digits between runs and the picture
    must not."""
    idx = {n: i for i, n in enumerate(G.nodes())}
    order = sorted(G.nodes(),
                   key=lambda z: (-round(float(w[idx[z]]), 9), -G.degree(z), z))
    n = len(order)
    seed_pos = {}
    for t, node in enumerate(order):             # a golden-angle spiral
        r = np.sqrt(t / max(n - 1, 1))
        a = t * np.pi * (3.0 - np.sqrt(5.0))
        seed_pos[node] = np.array([r * np.cos(a), r * np.sin(a)])
    pos = nx.spring_layout(G, pos=seed_pos, seed=3, iterations=900,
                           k=2.8 / np.sqrt(n))
    r = np.array([np.hypot(*pos[z]) for z in G.nodes()])
    # Scale on the typical radius rather than on the outermost element. A ring
    # puts every element at its maximum radius while a blob puts only a few
    # there, so normalising on the maximum makes the ring look larger than the
    # others when it is not. The geometric mean of the root mean square and the
    # maximum radius is the compromise: it shrinks the ring and leaves the
    # spread of the other three almost unchanged.
    s = np.sqrt(np.sqrt((r ** 2).mean()) * r.max()) or 1.0
    return {z: (v[0] / s, v[1] / s) for z, v in pos.items()}


def layout(kind, G, w):
    """One rule for all four, so every difference between the four pictures
    comes from the graphs rather than from the drawing."""
    return spread(G, w)


def ringdist(u, v):
    d = abs(u - v)
    return min(d, N - d)'''

In [4]:
# fig4_publication_3, cell 8: the released data and the measures, code unchanged.
SRC_LOAD = r'''import numpy as np, pandas as pd, json, warnings
warnings.filterwarnings('ignore')
from scipy import stats

RNG = np.random.default_rng(20260803)
B = 10000

# ------------------------------------------------------------------ measures
def share(x):
    x = np.asarray(x, float).ravel(); return x / x.sum()
def Hf(p):
    p = np.asarray(p, float); p = p[p > 0]; return float(-(p * np.log2(p)).sum())
def Jf(p): return Hf(p) / np.log2(len(p))
def comp(p): return 1 - Jf(p)
def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x); i = np.arange(1, n + 1)
    return float((2 * (i * x).sum()) / (n * x.sum()) - (n + 1) / n)
def eig_shares(A):
    w, v = np.linalg.eigh(A)
    u = np.abs(v[:, np.argmax(w)])
    return share(u)

# ------------------------------------------------------------------ find the files
# Leave DATA_DIR as None to search the usual places, or set it to the folder that
# holds the seven CSVs, for example DATA_DIR = r"C:\Users\me\Downloads\fig4_data".
DATA_DIR = None

FILES = ['team_sport_position_centrality.csv',
         'musculoskeletal_coupling_shares.csv',
         'musculoskeletal_random_null_shares.csv',
         'cortical_coupling_shares.csv',
         'cortical_threshold_sweep.csv',
         'expertise_outcome_entropy.csv',
         'cov_MOTOR_matrix.csv']

import io
from pathlib import Path

def _match(folder, name):
    """The file, allowing an arbitrary prefix such as a download hash."""
    p = folder / name
    if p.exists():
        return p
    hits = sorted(folder.glob('*' + name))
    return hits[0] if hits else None

def _locate():
    here = Path.cwd()
    cands = ([Path(DATA_DIR).expanduser()] if DATA_DIR else
             [here / 'data', here, here.parent / 'data', here.parent,
              Path.home() / 'Downloads' / 'data', Path.home() / 'Downloads'])
    best, best_missing = None, list(FILES)
    for d in cands:
        try:
            if not d.is_dir():
                continue
        except OSError:
            continue
        missing = [f for f in FILES if _match(d, f) is None]
        if not missing:
            return d
        if len(missing) < len(best_missing):
            best, best_missing = d, missing
    msg = ['The seven source files were not found.', '', 'Looked in:']
    msg += ['    ' + str(d) for d in cands]
    if best is not None:
        msg += ['', 'Closest match: ' + str(best), 'still missing:']
        msg += ['    ' + f for f in best_missing]
    msg += ['', 'Set DATA_DIR at the top of this cell to the folder that holds them.']
    raise FileNotFoundError('\n'.join(msg))

DATA = _locate()
print('reading from', DATA)

def rd(name):
    txt = _match(DATA, name).read_text(encoding='utf-8-sig')
    lines = [ln for ln in txt.splitlines() if not ln.lstrip().lstrip('"').startswith('#')]
    return pd.read_csv(io.StringIO('\n'.join(lines)))

def rmat(name):
    txt = _match(DATA, name).read_text(encoding='utf-8-sig')
    lines = [ln for ln in txt.splitlines()
             if ln.strip() and not ln.lstrip().lstrip('"').startswith('#')]
    return np.array([[float(v) for v in ln.split(',')] for ln in lines])

# ------------------------------------------------------------------ load
cent = rd('team_sport_position_centrality.csv')
mus = rd('musculoskeletal_coupling_shares.csv')
musnull = rd('musculoskeletal_random_null_shares.csv')
cort_sh = rd('cortical_coupling_shares.csv')
sweep = rd('cortical_threshold_sweep.csv')
exp = rd('expertise_outcome_entropy.csv')
M = rmat('cov_MOTOR_matrix.csv')

PUB = dict(football_H=3.3875, handball_H=2.2829, basket_H=2.2314, basket_sd=0.0477,
           basket_profile_H=2.2620, cort_H=6.5468, mus_H=7.7369, mus_null_H=7.9876,
           nov=3.316, exps=2.021, diff=1.295, mus_ratio=3.81,
           cort_comp=0.0146, mus_comp=0.0421, basket_comp=0.0390,
           football_comp=0.0208, handball_comp=0.1168)'''

In [5]:
# fig4_publication_3, cell 9: stage 1, every published value reproduced from the released files.
SRC_STAGE1 = r'''print('=' * 78)
print('STAGE 1  reproduction of every published value from the released files')
print('=' * 78)
CHK = []
def chk(name, got, pub, tol=5e-4):
    ok = abs(got - pub) <= tol
    CHK.append(ok)
    print(f'  {name:<42} computed {got:>10.4f}   published {pub:>10.4f}   {"OK" if ok else "MISMATCH"}')
    return got

# --- team sport
def sport_shares(sport, unit=None):
    q = cent[cent.sport == sport]
    if unit is not None: q = q[q.unit == unit]
    return share(q.centrality.values)

p_fb = sport_shares('Football'); p_hb = sport_shares('Handball')
chk('football H (bits)', Hf(p_fb), PUB['football_H'])
chk('handball H (bits)', Hf(p_hb), PUB['handball_H'])
chk('football compression', comp(p_fb), PUB['football_comp'])
chk('handball compression', comp(p_hb), PUB['handball_comp'])

teams = sorted(cent[cent.sport == 'Basketball'].unit.unique())
POS = ['PG', 'SG', 'SF', 'PF', 'CN']
prof = np.array([[cent[(cent.sport == 'Basketball') & (cent.unit == t) & (cent.position == q)].centrality.values[0]
                  for q in POS] for t in teams])
p_team = np.array([share(r) for r in prof])
H_team = np.array([Hf(p) for p in p_team])
chk('basketball H, mean of sixteen teams', H_team.mean(), PUB['basket_H'])
chk('basketball SD across teams', H_team.std(ddof=1), PUB['basket_sd'], 5e-4)
p_bk_mean = share(prof.mean(axis=0))   # average the raw centralities, then normalise
chk('basketball H of the averaged profile', Hf(p_bk_mean), PUB['basket_profile_H'])
comp_team = np.array([comp(p) for p in p_team])
chk('basketball compression, mean of teams', comp_team.mean(), PUB['basket_comp'])

# --- cortex, from the released matrix
A = np.abs(M.copy()); np.fill_diagonal(A, 0.0)
p_cx = eig_shares(A)
chk('cortical H from cov_MOTOR (bits)', Hf(p_cx), PUB['cort_H'])
chk('cortical compression', comp(p_cx), PUB['cort_comp'])
chk('cortical H from the shares file', Hf(cort_sh.share_p_i_eigenvector.values), PUB['cort_H'])

# --- muscles
p_ms = share(mus.bones_linked.values)
p_ms_null = share(musnull.bones_linked.values)
chk('musculoskeletal H (bits)', Hf(p_ms), PUB['mus_H'])
chk('musculoskeletal null H (bits)', Hf(p_ms_null), PUB['mus_null_H'])
chk('musculoskeletal compression', comp(p_ms), PUB['mus_comp'])
ratio = comp(p_ms) / comp(p_ms_null)
chk('musculoskeletal against its own null, ratio', ratio, PUB['mus_ratio'], 6e-3)

# --- expertise
nov = exp[exp.group == 'Novice'].outcome_entropy_h_bits.values
exs = exp[exp.group == 'Expert'].outcome_entropy_h_bits.values
chk('novice outcome entropy (bits)', nov.mean(), PUB['nov'])
chk('expert outcome entropy (bits)', exs.mean(), PUB['exps'])
chk('novice minus expert difference (bits)', nov.mean() - exs.mean(), PUB['diff'])

assert all(CHK), 'a published value did not reproduce, nothing further is computed'
print('\n  all values reproduce. proceeding.\n')

OUT = {}'''

In [6]:
# fig4_publication_3, cell 10: stage 2, resampling on the released data.
SRC_STAGE2 = r'''# =========================================================== STAGE 2 resampling
print('=' * 78)
print('STAGE 2  resampling on the released data')
print('=' * 78)

def boot(vals, stat, n=B):
    idx = RNG.integers(0, len(vals), size=(n, len(vals)))
    return np.array([stat(np.asarray(vals)[i]) for i in idx])

# --- A. basketball, bootstrap over the sixteen teams
bs_bk = np.array([comp_team[RNG.integers(0, 16, 16)].mean() for _ in range(B)])
ci_bk = np.percentile(bs_bk, [2.5, 97.5])
print(f'  basketball  mean compression {comp_team.mean():.4f}  bootstrap 95% CI '
      f'[{ci_bk[0]:.4f}, {ci_bk[1]:.4f}]  over 16 teams, {B:,} resamples')
print(f'              per-team range {comp_team.min():.4f} to {comp_team.max():.4f}, '
      f'all above zero: {bool((comp_team > 0).all())}')
OUT['basketball'] = dict(obs=float(comp_team.mean()), ci=[float(ci_bk[0]), float(ci_bk[1])],
                         boot=bs_bk.tolist(), per_team=comp_team.tolist(), teams=teams,
                         H_team=H_team.tolist(), lo=float(comp_team.min()), hi=float(comp_team.max()))

# --- B. cortex, bootstrap over regions of the released matrix
bs_cx = np.empty(2000)
for b in range(2000):
    idx = RNG.integers(0, 100, 100)
    Ab = A[np.ix_(idx, idx)]; np.fill_diagonal(Ab, 0.0)
    bs_cx[b] = comp(eig_shares(Ab))
ci_cx = np.percentile(bs_cx, [2.5, 97.5])
print(f'  cortex      compression {comp(p_cx):.4f}  bootstrap 95% CI '
      f'[{ci_cx[0]:.4f}, {ci_cx[1]:.4f}]  over 100 regions, 2,000 resamples')
print(f'              threshold sweep, compression from '
      f'{sweep.compression_1_minus_J.min():.4f} at all 4950 pairs to '
      f'{sweep.compression_1_minus_J.max():.4f} at the strongest 300')
OUT['cortex'] = dict(obs=float(comp(p_cx)), ci=[float(ci_cx[0]), float(ci_cx[1])],
                     boot=bs_cx.tolist(), sweep=sweep.to_dict('records'))

# --- C. muscles, bootstrap over the 270 muscles, against the released null
deg = mus.bones_linked.values.astype(float)
deg_null = musnull.bones_linked.values.astype(float)
bs_ms = np.array([comp(share(deg[RNG.integers(0, 270, 270)])) for _ in range(B)])
bs_ms_null = np.array([comp(share(deg_null[RNG.integers(0, 270, 270)])) for _ in range(B)])
bs_ratio = bs_ms / np.maximum(bs_ms_null, 1e-12)
ci_ms = np.percentile(bs_ms, [2.5, 97.5]); ci_msn = np.percentile(bs_ms_null, [2.5, 97.5])
ci_rt = np.percentile(bs_ratio, [2.5, 97.5])
print(f'  muscles     compression {comp(p_ms):.4f}  bootstrap 95% CI '
      f'[{ci_ms[0]:.4f}, {ci_ms[1]:.4f}]  over 270 muscles')
print(f'              released null {comp(p_ms_null):.4f}  95% CI [{ci_msn[0]:.4f}, {ci_msn[1]:.4f}]')
print(f'              ratio {ratio:.2f}  95% CI [{ci_rt[0]:.2f}, {ci_rt[1]:.2f}], '
      f'intervals overlap: {bool(ci_ms[0] < ci_msn[1])}')
OUT['muscles'] = dict(obs=float(comp(p_ms)), ci=[float(ci_ms[0]), float(ci_ms[1])],
                      boot=bs_ms.tolist(), null_obs=float(comp(p_ms_null)),
                      null_ci=[float(ci_msn[0]), float(ci_msn[1])], boot_null=bs_ms_null.tolist(),
                      ratio=float(ratio), ratio_ci=[float(ci_rt[0]), float(ci_rt[1])])

# --- D. expertise, bootstrap, permutation and a leave-one-out classifier
d_obs = nov.mean() - exs.mean()
bs_d = np.array([nov[RNG.integers(0, 10, 10)].mean() - exs[RNG.integers(0, 10, 10)].mean()
                 for _ in range(B)])
ci_d = np.percentile(bs_d, [2.5, 97.5])
pool = np.concatenate([nov, exs])
perm = np.empty(B)
for b in range(B):
    q = RNG.permutation(pool)
    perm[b] = q[:10].mean() - q[10:].mean()
p_perm = (1 + (np.abs(perm) >= abs(d_obs)).sum()) / (B + 1)
# leave-one-out nearest-threshold classifier on the single feature
y = np.r_[np.ones(10), np.zeros(10)]; x = pool
correct = 0
for i in range(20):
    m = np.ones(20, bool); m[i] = False
    thr = (x[m][y[m] == 1].mean() + x[m][y[m] == 0].mean()) / 2
    correct += int((x[i] > thr) == (y[i] == 1))
acc = correct / 20
accs_null = []
for b in range(2000):
    yp = RNG.permutation(y); c = 0
    for i in range(20):
        m = np.ones(20, bool); m[i] = False
        thr = (x[m][yp[m] == 1].mean() + x[m][yp[m] == 0].mean()) / 2
        c += int((x[i] > thr) == (yp[i] == 1))
    accs_null.append(c / 20)
accs_null = np.array(accs_null)
p_acc = (1 + (accs_null >= acc).sum()) / (len(accs_null) + 1)
print(f'  expertise   difference {d_obs:.3f} bits  bootstrap 95% CI [{ci_d[0]:.3f}, {ci_d[1]:.3f}]')
print(f'              permutation p = {p_perm:.1e} over {B:,} label shuffles')
print(f'              leave-one-out accuracy from outcome entropy alone {acc*100:.0f}%, '
      f'permuted mean {accs_null.mean()*100:.1f}%, p = {p_acc:.1e}')
print(f'              predictability ratio 2^diff = {2**d_obs:.3f}, '
      f'95% CI [{2**ci_d[0]:.3f}, {2**ci_d[1]:.3f}]')
OUT['expertise'] = dict(obs=float(d_obs), ci=[float(ci_d[0]), float(ci_d[1])], boot=bs_d.tolist(),
                        perm=perm.tolist(), p_perm=float(p_perm), acc=float(acc),
                        acc_null_mean=float(accs_null.mean()), p_acc=float(p_acc),
                        nov=nov.tolist(), exp=exs.tolist(),
                        ratio=float(2 ** d_obs), ratio_ci=[float(2 ** ci_d[0]), float(2 ** ci_d[1])])

# --- Gini, a second reading
print('\n  Gini of the coupling shares, alongside compression')
GINI = {}
for nm, p in (('Football', p_fb), ('Handball', p_hb), ('Basketball (mean profile)', p_bk_mean),
              ('Cortical', p_cx), ('Musculoskeletal', p_ms), ('Musculoskeletal null', p_ms_null)):
    GINI[nm] = dict(gini=gini(p), comp=comp(p), H=Hf(p), N=len(p))
    print(f'    {nm:<28} N {len(p):>4}   Gini {gini(p):.4f}   compression {comp(p):.4f}')
OUT['gini'] = GINI
OUT['sports'] = dict(football=dict(H=Hf(p_fb), comp=comp(p_fb), gini=gini(p_fb), N=11),
                     handball=dict(H=Hf(p_hb), comp=comp(p_hb), gini=gini(p_hb), N=6))

V = OUT
print('\nready to draw')'''

In [7]:
import hashlib

def load_cells(name, srcs, cwd=None):
    m = types.ModuleType(name)
    m.__file__ = name + '.py'
    here, buf = os.getcwd(), io.StringIO()
    try:
        if cwd:
            os.chdir(cwd)
        with contextlib.redirect_stdout(buf):
            for s in srcs:
                exec(compile(s, name + '.py', 'exec'), m.__dict__)
    finally:
        os.chdir(here)
    return m, buf.getvalue()

for k, s in [('core', SRC_CORE), ('ensembles', SRC_ENS), ('data', SRC_LOAD), ('stage 1', SRC_STAGE1),
             ('stage 2', SRC_STAGE2)]:
    print(f'{k:<10} sha256 {hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]}')
P2E, _ = load_cells('p2_fig3_ensembles', [SRC_CORE, SRC_ENS])
P2M, P2M_OUT = load_cells('p2_fig4_measured', [SRC_LOAD, SRC_STAGE1, SRC_STAGE2], cwd=P2_DIR)
print('core', P2E.CORE_VERSION, '| NetworkX', nx.__version__, '| NumPy', np.__version__)
print('ensembles of', P2E.N, 'elements and', P2E.M_LINKS, 'links,', P2E.N_REAL, 'realizations each, seeds',
      P2E.SEED0, 'to', P2E.SEED0 + P2E.N_REAL - 1)
print('check, the published values of the second paper reproduce from its released files:', all(P2M.CHK))

core       sha256 7624eea494ef5d8d
ensembles  sha256 f9f4b795bd0df9ca
data       sha256 57f5f1bc4a1ececd
stage 1    sha256 5a04a86f3b8dd04f
stage 2    sha256 84690e9817952f17
core v5 / 24 July 2026 (Methods Eq 1-13) | NetworkX 3.7 | NumPy 2.5.3
ensembles of 100 elements and 300 links, 100 realizations each, seeds 42 to 141
check, the published values of the second paper reproduce from its released files: True


## Step 1. The four matched ensembles (panels A and C, and the scale of panel B)

In [8]:
CEIL, N = P2E.CEIL, P2E.N
ENS = {t: P2E.ens[KIND[t]] for t in TOPOS}
DH = {t: np.clip(CEIL - ENS[t]['H'], 0.0, None) for t in TOPOS}          # compression per realization
DH_DEG = {t: np.clip(CEIL - ENS[t]['Hdeg'], 0.0, None) for t in TOPOS}
CHECK = []

print(f'ceiling log2 N = {CEIL:.4f} bits')
print(f'{"":<12}{"H":>8}{"pub":>8}{"dH":>8}{"pub":>7}{"2^H":>7}{"pub":>7}{"kappa":>7}{"pub":>7}'
      f'{"H deg":>8}{"pub":>8}{"dH deg":>8}')
for t in TOPOS:
    e = ENS[t]
    H, Hd, k = e['H'].mean(), e['Hdeg'].mean(), e['kappa'].mean()
    row = [round(H, 4) == PUB['H'][t], round(max(CEIL - H, 0.0), 3) == PUB['dH'][t],
           round(2 ** H, 1) == PUB['deff'][t], round(k, 2) == PUB['kappa'][t],
           round(Hd, 4) == PUB['H_deg'][t], set(e['m']) == {P2E.M_LINKS}]
    CHECK += row
    print(f'{P2E.LABEL[KIND[t]]:<12}{H:>8.4f}{PUB["H"][t]:>8.4f}{max(CEIL - H, 0):>8.3f}{PUB["dH"][t]:>7.3f}'
          f'{2 ** H:>7.1f}{PUB["deff"][t]:>7.1f}{k:>7.2f}{PUB["kappa"][t]:>7.2f}{Hd:>8.4f}{PUB["H_deg"][t]:>8.4f}'
          f'{max(CEIL - Hd, 0):>8.4f}')
print()
for t in TOPOS:
    lo, hi = P2E.CI[KIND[t]]
    print(f'{P2E.LABEL[KIND[t]]:<16} H 95% interval [{lo:.4f}, {hi:.4f}], compression over realizations '
          f'{DH[t].min():.3f} to {DH[t].max():.3f}, kappa {ENS[t]["kappa"].min():.2f} to {ENS[t]["kappa"].max():.2f}')
print()
print('check, Table 1 of the second paper regenerated, every realization at 300 links:', all(CHECK))

ceiling log2 N = 6.6439 bits
                   H     pub      dH    pub    2^H    pub  kappa    pub   H deg     pub  dH deg
Regular lattice  6.6439  6.6439   0.000  0.000  100.0  100.0   6.00   6.00  6.6439  6.6439  0.0000
Small-world   6.5832  6.5832   0.061  0.061   95.9   95.9   6.09   6.09  6.6324  6.6324  0.0114
Random        6.4820  6.4820   0.162  0.162   89.4   89.4   6.90   6.90  6.5313  6.5313  0.1126
Scale-free    6.3176  6.3176   0.326  0.326   79.8   79.8  10.10  10.10  6.2865  6.2865  0.3574

Regular lattice  H 95% interval [6.6439, 6.6439], compression over realizations 0.000 to 0.000, kappa 6.00 to 6.00
Small-world      H 95% interval [6.5768, 6.5889], compression over realizations 0.024 to 0.231, kappa 6.05 to 6.15
Random           H 95% interval [6.4772, 6.4867], compression over realizations 0.083 to 0.229, kappa 6.54 to 7.20
Scale-free       H 95% interval [6.3151, 6.3201], compression over realizations 0.306 to 0.366, kappa 8.96 to 11.13

check, Table 1 of the sec

## Step 2. The three measured structures and the outcome (panel B)

In [9]:
V = P2M.V
CX_N = 100
CX_H = float(P2M.Hf(P2M.p_cx))
CX_DH = np.log2(CX_N) - CX_H
CX_CI = [c * np.log2(CX_N) for c in V['cortex']['ci']]
RCX = np.loadtxt(os.path.join(P2_DIR, 'cov_MOTOR_matrix.csv'), delimiter=',', comments='#')
WCX = np.abs(RCX); np.fill_diagonal(WCX, 0.0)
P_CX = np.asarray(P2M.p_cx, float)

def eig_shares(W):
    v = np.abs(np.linalg.eigh(W)[1][:, -1])
    return v / v.sum()

print(f'cortex: N {CX_N}, H {CX_H:.4f} bits (published 6.5468), compression {CX_DH:.4f} bits, '
      f'95% interval [{CX_CI[0]:.4f}, {CX_CI[1]:.4f}]')
print('the shares of the second paper equal the eigenvector shares of |r|:', np.allclose(P_CX, eig_shares(WCX)))

SCH = os.path.join(F3_DIR, 'schaefer2018')
iu = np.triu_indices(CX_N, 1)
rng = np.random.default_rng(0)
ATLAS = {}
for tag, fname in [('17-network', 'Schaefer2018_100Parcels_17Networks_order_FSLMNI152_2mm.Centroid_RAS.csv'),
                   ('7-network', 'Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv'),
                   ('Kong 2022', 'Schaefer2018_100Parcels_Kong2022_17Networks_order_FSLMNI152_2mm.Centroid_RAS.csv')]:
    C = pd.read_csv(os.path.join(SCH, fname)); xyz = C[['R', 'A', 'S']].values.astype(float)
    D = np.linalg.norm(xyz[:, None] - xyz[None], axis=2)
    rho = spearmanr(D[iu], RCX[iu]).correlation
    mirror = np.linalg.norm(xyz[:, None] - (xyz * [-1, 1, 1])[None], axis=2)
    partner = 50 + np.argmin(mirror[:50, 50:], axis=1)
    homo = RCX[np.arange(50), partner].mean()
    net = C['ROI Name'].str.split('_').str[2].values
    same = (net[:, None] == net[None])[iu]
    null = np.array([spearmanr(D[iu], RCX[np.ix_(p, p)][iu]).correlation for p in (rng.permutation(CX_N) for _ in range(300))])
    ATLAS[tag] = dict(rho=rho, null=null.mean(), sd=null.std(), homo=homo, same=RCX[iu][same].mean(),
                      other=RCX[iu][~same].mean(), xyz=xyz, net=net, partner=partner)
    print(f'{tag:<11} distance and coupling, Spearman {rho:+.3f} (shuffled labels {null.mean():+.3f}, sd {null.std():.3f}); '
          f'mirror-image pairs r {homo:.3f} against {RCX[iu].mean():.3f} for all pairs; same network r '
          f'{RCX[iu][same].mean():.3f}, other {RCX[iu][~same].mean():.3f}')
A17 = ATLAS['17-network']
CHECK_ATLAS = (A17['rho'] < A17['null'] - 10 * A17['sd'] and
               all(A17['rho'] < ATLAS[t]['rho'] and A17['homo'] > ATLAS[t]['homo'] for t in ('7-network', 'Kong 2022')))
XY_CX = A17['xyz'][:, :2]                 # right and anterior, millimetres, for the drawing
print('check, the matrix follows the 17-network order of the Schaefer 100-parcel atlas:', CHECK_ATLAS)

cortex: N 100, H 6.5468 bits (published 6.5468), compression 0.0971 bits, 95% interval [0.0686, 0.1260]
the shares of the second paper equal the eigenvector shares of |r|: True
17-network  distance and coupling, Spearman -0.340 (shuffled labels +0.001, sd 0.015); mirror-image pairs r 0.451 against 0.020 for all pairs; same network r 0.359, other -0.000
7-network   distance and coupling, Spearman -0.191 (shuffled labels -0.000, sd 0.015); mirror-image pairs r 0.165 against 0.020 for all pairs; same network r 0.194, other -0.012
Kong 2022   distance and coupling, Spearman -0.076 (shuffled labels -0.001, sd 0.016); mirror-image pairs r 0.116 against 0.020 for all pairs; same network r 0.223, other 0.008
check, the matrix follows the 17-network order of the Schaefer 100-parcel atlas: True


### Step 2b. The musculoskeletal map.

In [10]:
import openpyxl

MUR = os.path.join(F3_DIR, 'murphy2018')
INC = pd.read_csv(os.path.join(MUR, 'incidence_matrix.csv'), index_col=0)
B = (INC.values > 0).astype(float)
AMS = ((B.T @ B) > 0).astype(float); np.fill_diagonal(AMS, 0.0)

def comp(W, kind='eig'):
    G = nx.from_numpy_array(W)
    cc = sorted(max(nx.connected_components(G), key=len))
    Wc = W[np.ix_(cc, cc)]
    p = eig_shares(Wc) if kind == 'eig' else Wc.sum(1) / Wc.sum()
    H = -(p[p > 0] * np.log2(p[p > 0])).sum()
    return len(cc), int((Wc > 0).sum() / 2), H, np.log2(len(cc)) - H

def workbook_table(sheet):
    wb = openpyxl.load_workbook(os.path.join(MUR, 'supplementaryData.xlsx'), read_only=True, data_only=True)
    rows = [list(r) for r in wb[sheet].iter_rows(values_only=True)]
    hdr = rows[2]; ncol = max(i for i, x in enumerate(hdr) if x not in (None, '', ' ')) + 1
    body = [r for r in rows[3:] if r[1] not in (None, '', ' ', 'Hyperedge degree')]
    return np.array([[1.0 if (x not in (None, '', ' ') and float(x) > 0) else 0.0 for x in r[2:ncol]] for r in body])

S9, S10 = workbook_table('Table S9'), workbook_table('Table S10')
CHECK_S9 = S9.shape == B.shape and np.array_equal(S9, B)
MS_N, MS_L, MS_H, MS_DH = comp(AMS)
_, _, MS_HD, MS_DHD = comp(AMS, 'deg')
A10 = ((S10.T @ S10) > 0).astype(float); np.fill_diagonal(A10, 0.0)
M10_N, M10_L, M10_H, M10_DH = comp(A10)
P_MS = eig_shares(AMS)
print(f'incidence matrix: {B.shape[0]} bones, {B.shape[1]} muscles, {int(B.sum())} attachments')
print(f'muscles linked by a shared bone: N {MS_N}, {MS_L} links, H {MS_H:.4f} bits, compression {MS_DH:.4f} bits '
      f'on eigenvector shares and {MS_DHD:.4f} bits on degree shares')
print(f'Table S10, the second atlas: N {M10_N} in the largest component, {M10_L} links, compression {M10_DH:.4f} bits')
top = np.argsort(-P_MS)[:6]
print('largest shares:', ', '.join(f'{INC.columns[i]} {P_MS[i] * MS_N:.2f}' for i in top), '(times an even share)')
print('check, Table S9 of the workbook equals the released incidence matrix:', CHECK_S9)

incidence matrix: 173 bones, 270 muscles, 1010 attachments
muscles linked by a shared bone: N 270, 2679 links, H 7.1466 bits, compression 0.9302 bits on eigenvector shares and 0.3104 bits on degree shares
Table S10, the second atlas: N 267 in the largest component, 2835 links, compression 0.6628 bits
largest shares: Longissimus Thoracis 5.94, Trapezius 5.37, Multifidus 4.83, Iliocostalis Cervicis 4.30, Semispinalis Thoracis 4.04, Splenius Capitis 4.02 (times an even share)
check, Table S9 of the workbook equals the released incidence matrix: True


### Step 2c. The muscles of a badminton stroke.

In [11]:
MSB = os.path.join(F3_DIR, 'multisensebadminton')
ANN = pd.read_excel(os.path.join(MSB, 'documentation', 'Annotation Data File.xlsx'))
ANN.columns = ['sub', 't0', 't1', 'num', 'type', 'skill', 'h', 'v', 'hit', 'sound', 'h1', 'h2', 'h3', 's1', 's2', 's3']

def fisher_mean(rs):
    z = np.arctanh(np.clip(np.array(rs, float), -0.999999, 0.999999))
    return np.tanh(np.nanmean(z, axis=0))

def dH_of(W, kind='eig'):
    W = np.array(W, float); np.fill_diagonal(W, 0.0)
    p = eig_shares(W) if kind == 'eig' else W.sum(1) / W.sum()
    return np.log2(len(W)) + (p[p > 0] * np.log2(p[p > 0])).sum()

def entropy_of(labels):
    c = pd.Series(labels).value_counts().values.astype(float); p = c / c.sum()
    return -(p * np.log2(p)).sum()

REC, W_E, W_J, RATES = [], [], [], []
for f in sorted(glob.glob(os.path.join(MSB, 'coupling', '*.npz'))):
    d = np.load(f, allow_pickle=True)
    if int(d['n']) == 0:
        continue
    sub, typ = str(d['sub']), str(d['type'])
    WE_r, WJ_r = np.abs(fisher_mean(d['RE'])), np.abs(fisher_mean(d['RJ']))
    g = ANN[(ANN['sub'] == sub) & (ANN['type'] == typ)]
    g = g[(g.h.astype(str) != 'NoVid') & (g.v.astype(str) != 'NoVid')]
    REC.append(dict(sub=sub, type='clear' if typ.startswith('Fore') else 'drive', skill=str(d['skill']),
                    n=int(d['n']), dHE=dH_of(WE_r), dHJ=dH_of(WJ_r), dHEdeg=dH_of(WE_r, 'deg'),
                    dHJdeg=dH_of(WJ_r, 'deg'),
                    Hout=entropy_of(g.h.astype(str) + '|' + g.v.astype(str)),
                    Hland=entropy_of((lambda q: q.h.astype(str) + '|' + q.v.astype(str))(
                        g[~g.h.astype(str).isin(['Contact', 'Not Contact'])])),
                    file=os.path.basename(f), RE=d['RE'], RJ=d['RJ']))
    W_E.append(WE_r); W_J.append(WJ_r); RATES.append(d['rates'])
T = pd.DataFrame([{k: v for k, v in r.items() if k not in ('RE', 'RJ')} for r in REC])
T['rank'] = T.skill.map({'Beginner': 0, 'Intermediate': 1, 'Expert': 2})
RATES = np.array(RATES); RATE = np.median(RATES, 0)
print(f'{len(T)} recordings of {T["sub"].nunique()} players, {T.n.sum()} strokes in use '
      f'({T[T.type == "clear"].n.sum()} clears, {T[T.type == "drive"].n.sum()} drives) of the {len(ANN)} annotated')
print('players per stroke and skill:', T.groupby(['type', 'skill']).size().to_dict())
print('sampling rates over the recordings, median (range): ' + ', '.join(
      f'{nm} {RATE[i]:.0f} Hz ({RATES[:, i].min():.0f} to {RATES[:, i].max():.0f})'
      for i, nm in enumerate(('thigh EMG', 'forearm EMG', 'upper-arm EMG', 'joints'))))

# bootstrap intervals over players, in a fixed order of draws that the outcome cell below continues
rng = np.random.default_rng(2026)
def boot_mean(df, col, B=4000):
    subs = df['sub'].unique()
    g = {s: df[df['sub'] == s][col].values for s in subs}
    m = [np.mean(np.concatenate([g[s] for s in rng.choice(subs, len(subs))])) for _ in range(B)]
    return float(df[col].mean()), [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]
ST = {}
for k in ('dHE', 'dHJ'):
    ST[k] = {'all': boot_mean(T, k)}
    for sk in ('Beginner', 'Intermediate', 'Expert'):
        ST[k][sk] = boot_mean(T[T.skill == sk], k)
    for ty in ('clear', 'drive'):
        ST[k][ty] = boot_mean(T[T.type == ty], k)
    p = T.pivot_table(index='sub', columns='type', values=k).dropna()
    d = (p['clear'] - p['drive']).values
    bs = [rng.choice(d, d.size).mean() for _ in range(4000)]
    ST[k]['task'] = [float(d.mean()), [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))], int(d.size)]
    for ty in ('clear', 'drive'):
        t = T[T.type == ty]
        rho = spearmanr(t['rank'], t[k]).correlation
        perm = np.array([spearmanr(rng.permutation(t['rank'].values), t[k]).correlation for _ in range(4000)])
        ST[k]['rho_' + ty] = [float(rho), float((np.abs(perm) >= abs(rho)).mean())]

# the label-shuffle control, recording by recording, on a generator of its own
rc = np.random.default_rng(7)
CTRL = {'E': [], 'J': []}
for r in REC:
    for k, R in (('E', r['RE']), ('J', r['RJ'])):
        n = R.shape[1]; nulls = []
        for _ in range(50):
            Rp = np.array([x[np.ix_(pm, pm)] for x, pm in ((x, rc.permutation(n)) for x in R)])
            nulls.append(dH_of(np.abs(fisher_mean(Rp))))
        CTRL[k].append((np.mean(nulls), np.percentile(nulls, 95)))
CTRL = {k: np.array(v) for k, v in CTRL.items()}
CHECK_EMG = bool(np.all(T.dHE.values > CTRL['E'][:, 1]) and np.all(T.dHJ.values > CTRL['J'][:, 1]))

# the pooled network of the 20 sites, for the drawing and for the groups
WE = np.mean(W_E, 0); np.fill_diagonal(WE, 0.0)
P_E = eig_shares(WE)
GROUPS = [list(range(0, 8)), list(range(8, 16)), list(range(16, 20))]
within = [WE[np.ix_(g, g)][np.triu_indices(len(g), 1)].mean() for g in GROUPS]
between = np.mean([WE[np.ix_(a, b)].mean() for a, b in ((GROUPS[0], GROUPS[1]), (GROUPS[0], GROUPS[2]), (GROUPS[1], GROUPS[2]))])

E_ = ST['dHE']
print(f'muscles: compression {E_["all"][0]:.4f} bits, 95% interval [{E_["all"][1][0]:.4f}, {E_["all"][1][1]:.4f}]; '
      f'label-shuffle control {CTRL["E"][:, 0].mean():.4f} bits, every recording above the 95th percentile of its own')
print(f'  coupling within the forearm band {within[0]:.3f}, the upper-arm band {within[1]:.3f}, the thigh {within[2]:.3f}, '
      f'between groups {between:.3f}; shares of the pooled network {P_E.min() * 20:.2f} to {P_E.max() * 20:.2f} times even')
for sk in ('Beginner', 'Intermediate', 'Expert'):
    print(f'  {sk.lower():<12} {E_[sk][0]:.4f} [{E_[sk][1][0]:.4f}, {E_[sk][1][1]:.4f}]')
print(f'  skill and compression, Spearman: clear {E_["rho_clear"][0]:+.3f} (p {E_["rho_clear"][1]:.2f}), '
      f'drive {E_["rho_drive"][0]:+.3f} (p {E_["rho_drive"][1]:.2f}); clear minus drive within player '
      f'{E_["task"][0]:+.4f} [{E_["task"][1][0]:+.4f}, {E_["task"][1][1]:+.4f}]')
J_ = ST['dHJ']
print(f'joints: compression {J_["all"][0]:.4f} bits [{J_["all"][1][0]:.4f}, {J_["all"][1][1]:.4f}], control '
      f'{CTRL["J"][:, 0].mean():.4f}; beginners {J_["Beginner"][0]:.4f}, intermediates {J_["Intermediate"][0]:.4f}, '
      f'experts {J_["Expert"][0]:.4f}; Spearman clear {J_["rho_clear"][0]:+.3f} (p {J_["rho_clear"][1]:.2f}), drive '
      f'{J_["rho_drive"][0]:+.3f} (p {J_["rho_drive"][1]:.2f}); clear minus drive {J_["task"][0]:+.4f} '
      f'[{J_["task"][1][0]:+.4f}, {J_["task"][1][1]:+.4f}]')
print('check, every recording couples its sites and its joints above its label-shuffle control:', CHECK_EMG)

50 recordings of 25 players, 7634 strokes in use (3798 clears, 3836 drives) of the 7763 annotated
players per stroke and skill: {('clear', 'Beginner'): 11, ('clear', 'Expert'): 6, ('clear', 'Intermediate'): 8, ('drive', 'Beginner'): 11, ('drive', 'Expert'): 5, ('drive', 'Intermediate'): 9}
sampling rates over the recordings, median (range): thigh EMG 502 Hz (444 to 502), forearm EMG 96 Hz (50 to 101), upper-arm EMG 115 Hz (54 to 198), joints 61 Hz (56 to 63)
muscles: compression 0.1706 bits, 95% interval [0.1372, 0.2069]; label-shuffle control 0.0031 bits, every recording above the 95th percentile of its own
  coupling within the forearm band 0.462, the upper-arm band 0.451, the thigh 0.399, between groups 0.088; shares of the pooled network 0.48 to 1.24 times even
  beginner     0.1508 [0.1189, 0.1895]
  intermediate 0.1946 [0.1207, 0.2848]
  expert       0.1730 [0.1160, 0.2414]
  skill and compression, Spearman: clear +0.083 (p 0.69), drive +0.096 (p 0.65); clear minus drive within p

### Step 2d. The outcome.

In [12]:
# the draws continue the generator of the cell above, so the cells run in order
OUT = {}
for ty in ('clear', 'drive'):
    t = T[T.type == ty]
    b, e = t[t.skill == 'Beginner'].Hout.values, t[t.skill == 'Expert'].Hout.values
    obs = b.mean() - e.mean()
    bs = [rng.choice(b, b.size).mean() - rng.choice(e, e.size).mean() for _ in range(10000)]
    pool = np.r_[b, e]; reach = 0
    for _ in range(10000):
        q = rng.permutation(pool); reach += abs(q[:b.size].mean() - q[b.size:].mean()) >= abs(obs)
    bl, el = t[t.skill == 'Beginner'].Hland.values, t[t.skill == 'Expert'].Hland.values
    OUT[ty] = dict(obs=float(obs), ci=[float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],
                   reach=int(reach), beg=float(b.mean()), exp=float(e.mean()),
                   inter=float(t[t.skill == 'Intermediate'].Hout.mean()), nb=b.size, ne=e.size,
                   ni=int((t.skill == 'Intermediate').sum()), landed=float(bl.mean() - el.mean()))
    o = OUT[ty]
    print(f'{ty}: beginners {o["beg"]:.3f}, intermediates {o["inter"]:.3f}, experts {o["exp"]:.3f} bits; '
          f'difference {o["obs"]:.3f} [{o["ci"][0]:.3f}, {o["ci"][1]:.3f}], {o["reach"]} of 10,000 label shuffles '
          f'as large; landed shots only {o["landed"]:.3f}')

clear: beginners 2.161, intermediates 1.641, experts 0.600 bits; difference 1.561 [1.116, 2.010], 0 of 10,000 label shuffles as large; landed shots only 1.338
drive: beginners 2.033, intermediates 1.944, experts 1.554 bits; difference 0.479 [0.045, 0.902], 929 of 10,000 label shuffles as large; landed shots only 0.632


## Step 3. The three panels, one figure each

In [13]:
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, ConnectionPatch, FancyArrowPatch, Rectangle
from scipy.interpolate import splev, splprep
from scipy.spatial import ConvexHull
import logging
logging.getLogger('matplotlib.axes._base').setLevel(logging.ERROR)   # equal aspect sets its own limits

mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.4,
                     'xtick.labelsize': 6.0, 'ytick.labelsize': 6.0, 'legend.fontsize': 6.0,
                     'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
                     'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False})

FIG_W = 6.93                            # the width of Fig. 2, inches
PW_S, PH = 1.48, 2.28                   # side panel width, and the height of every panel
PW_M = FIG_W - 2 * PW_S                 # the middle panel takes the rest of the row
BOX_S = (0.36, 0.33, 1.06, 1.65)        # axes box of a priced structure, y axis on the left
BOX_SR = (0.06, 0.33, 1.06, 1.65)       # the same box with the y axis on the right
BOX_L = (0.40, 0.33, 2.54, 1.65)        # the middle plot, bits saved by structures
BOX_R = (3.04, 0.33, 0.60, 1.65)        # the outcome entropy beside it, on its own scale

INK, NEUTRAL, LINK = '#1A1A1A', '#37474F', '#C3CBD0'
NODE_EDGE, SHELL_EDGE = '#37474F', '#B8C1C7'
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
VLIM = 2.7157337518864986               # the color scale of Fig. 2 and of the second paper, in bits
CNORM = Normalize(-VLIM, VLIM)
TITLE = {'A': 'nearest neighbors', 'B': 'bits saved by coordinative structure and by skill',
         'C': 'preferential attachment'}
SHOW_W = {t: np.sort(P2E.SHOW_W[KIND[t]])[::-1] for t in ('LAT', 'BA')}   # shares of the drawn realizations

def colour(p):
    b = np.log2(np.asarray(p, float) * len(p))
    return CMAP(CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))

def axes_at(fig, ox, oy, box):
    W, H = fig.get_size_inches()
    l, b, w, h = box
    return fig.add_axes([(ox + l) / W, (oy + b) / H, w / W, h / H])

def heading(fig, ox, oy, ph, letter, frames):
    # the title centered over the plot frames of the panel (ox + x0 to ox + x1, inches), the letter beside it
    W, H = fig.get_size_inches(); y = (oy + ph - 0.16) / H
    r = fig.canvas.get_renderer()
    t2 = fig.text(0, y, TITLE[letter], fontsize=7.9, color='k', ha='center', va='baseline')
    t2.set_x((ox + (frames[0] + frames[1]) / 2) / W)
    bb = t2.get_window_extent(r).transformed(fig.transFigure.inverted())
    fig.text(bb.x0 - 0.07 / W, y, letter, fontsize=9.4, fontweight='bold', color='k', ha='right', va='baseline')

def frame(ax, xt, yt=None):
    # a line above and below, none at the sides, dotted rules, ticks at the minimum, middle and maximum
    ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
    fmt = mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}')
    ax.xaxis.set_major_formatter(fmt); ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(fmt); ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        ax.grid(True, axis='y', alpha=0.40, linestyle=':', linewidth=0.5, color='gray')
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)

def vframe(ax, yt, right=False):
    # the same frame for a plot whose values run down the y axis
    ax.set_yticks(yt)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.grid(True, axis='y', alpha=0.40, linestyle=':', linewidth=0.5, color='gray'); ax.set_axisbelow(True)
    if right:
        ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=0, pad=2.0)

def draw_price(fig, ox, oy, t, letter):
    right = t == 'BA'                               # the right-hand panel carries its y axis on the right
    box = BOX_SR if right else BOX_S
    ax = axes_at(fig, ox, oy, box)
    frame(ax, [0, 0.5, 1], [0, 5, 10])
    if right:
        ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    w = SHOW_W[t]
    x0 = np.concatenate([[0.0], np.cumsum(w)[:-1]])
    h = np.log2(1.0 / w)
    ax.add_collection(PatchCollection([Rectangle((x0[i], 0.0), w[i], h[i]) for i in range(len(w))],
                                      facecolor=colour(w), edgecolor=colour(w), linewidth=0.15, zorder=2))
    ax.axhline(CEIL, color=INK, lw=0.7, ls=(0, (3.2, 2.2)), zorder=4)
    ax.plot([0, 1], [(w * h).sum()] * 2, color=INK, lw=0.9, zorder=5)
    ax.set_xlabel('cumulative share', labelpad=1.6)
    ax.set_ylabel(r'price  $\log_2(1/p_i)$  (bits)', labelpad=2.0)
    if t == 'LAT':
        ax.legend(handles=[Line2D([], [], color=INK, lw=0.7, ls=(0, (3.2, 2.2)), label=r'ceiling  $\log_2 N$'),
                           Line2D([], [], color=INK, lw=0.9, label=r'control entropy  $H$')],
                  loc='upper left', fontsize=5.8, handlelength=2.0, handletextpad=0.4, labelspacing=0.2,
                  borderaxespad=0.15)
    else:
        i_w, i_s = len(w) - 12, 2                   # a weaker element near the end, a stronger near the start
        ax.add_patch(FancyArrowPatch((x0[i_w] + w[i_w] / 2, h[i_w] + 0.3), (x0[i_s] + w[i_s] / 2, h[i_s] + 0.3),
                                     connectionstyle='arc3,rad=0.30', arrowstyle='-|>', mutation_scale=6,
                                     lw=0.7, color=INK, zorder=6))
        ax.text(0.52, 8.55, r'$\varepsilon$', fontsize=6.4, color='k', ha='center', va='bottom')
    heading(fig, ox, oy, PH, letter, (box[0], box[0] + box[2]))

# ---------------------------------------------------------------- the drawings of panel B
def outline(pts, pad):
    # a smooth closed curve round a set of points, pad millimetres outside their hull
    c = pts.mean(0); hull = pts[ConvexHull(pts).vertices]
    v = hull - c; hull = hull + pad * v / np.linalg.norm(v, axis=1)[:, None]
    tck, _ = splprep([np.r_[hull[:, 0], hull[0, 0]], np.r_[hull[:, 1], hull[0, 1]]], s=len(hull) * 6.0, per=True)
    return np.column_stack(splev(np.linspace(0, 1, 200), tck))

def size_of(q, smax, smin=1.2):
    return smin + smax * ((q - q.min()) / (q.max() - q.min() + 1e-12)) ** 0.7

def links(g, pos, pairs, strength, share, lw):
    # each link colored by the geometric mean share of its two ends, wider and darker the stronger it is
    n = len(share)
    m = np.array([np.sqrt(share[u] * share[v]) for u, v in pairs])
    s = np.asarray(strength, float); f = (s - s.min()) / (s.max() - s.min() + 1e-12)
    for k in np.argsort(f):
        u, v = pairs[k]
        g.plot(*zip(pos[u], pos[v]), color=CMAP(CNORM(np.log2(m[k] * n))), lw=lw[0] + lw[1] * f[k],
               alpha=0.35 + 0.5 * f[k], zorder=1 + f[k], solid_capstyle='round')

def dots(g, pos, share, smax):
    ids = np.argsort(share); q = share[ids]
    g.scatter([pos[i][0] for i in ids], [pos[i][1] for i in ids], s=size_of(q, smax),
              c=CMAP(CNORM(np.log2(q * len(share)))), edgecolors=NODE_EDGE, linewidths=0.22, alpha=0.92,
              zorder=3, clip_on=False)

def draw_cortex(g):
    pos = {i: XY_CX[i] for i in range(CX_N)}
    for hemi in (XY_CX[:, 0] < 0, XY_CX[:, 0] > 0):
        o = outline(XY_CX[hemi], 9.0)
        g.plot(o[:, 0], o[:, 1], color=SHELL_EDGE, lw=0.5, zorder=0.5)
    pairs = sorted({tuple(sorted((i, int(np.argmax(WCX[i]))))) for i in range(CX_N)})
    links(g, pos, pairs, [WCX[u, v] for u, v in pairs], P_CX, (0.3, 1.2))
    dots(g, pos, P_CX, 6.5)
    g.set_xlim(-82, 82); g.set_ylim(-116, 82); g.set_aspect('equal', adjustable='datalim'); g.axis('off')
    return pos

def draw_muscles(g):
    pos = {}
    for grp, (cx, cy), r in zip(GROUPS, [(-0.60, -0.50), (-0.60, 0.50), (0.68, 0.0)], [0.32, 0.32, 0.21]):
        for k, i in enumerate(grp):
            th = np.pi / 2 - 2 * np.pi * k / len(grp)
            pos[i] = (cx + r * np.cos(th), cy + r * np.sin(th))
    pairs = set()
    for grp in GROUPS:
        for i in grp:
            for j in sorted(grp, key=lambda z: -WE[i, z])[:2]:
                if j != i:
                    pairs.add(tuple(sorted((i, j))))
    for a_ in range(3):
        for b_ in range(a_ + 1, 3):
            pairs.add(max(((i, j) for i in GROUPS[a_] for j in GROUPS[b_]), key=lambda e: WE[e]))
    pairs = sorted(pairs)
    links(g, pos, pairs, [WE[u, v] for u, v in pairs], P_E, (0.35, 1.6))
    dots(g, pos, P_E, 12.0)
    g.set_xlim(-1.08, 1.04); g.set_ylim(-0.92, 0.92); g.set_aspect('equal', adjustable='datalim'); g.axis('off')
    return pos

def tree_from_root(G, share):
    # the links left when the graph is thinned from the element of largest share outward, each element tied to
    # the neighbor of largest share one step nearer the root, as the second paper draws its networks
    G = G.subgraph(max(nx.connected_components(G), key=lambda c: (len(c), sorted(c)[0]))).copy()
    root = max(G.nodes(), key=lambda z: (share[z], z))
    dep = nx.single_source_shortest_path_length(G, root)
    T_ = nx.Graph(); T_.add_nodes_from(sorted(G.nodes())); seen = {root}
    for u in sorted(G.nodes(), key=lambda z: (dep[z], -share[z], z)):
        for v in sorted(G.neighbors(u), key=lambda z: (-share[z], z)):
            if v not in seen and dep[v] == dep[u] + 1:
                seen.add(v); T_.add_edge(u, v)
    return T_

_T = tree_from_root(nx.from_numpy_array(AMS), P_MS)
_T2 = nx.Graph(); _T2.add_nodes_from(sorted(_T.nodes())); _T2.add_edges_from(sorted(tuple(sorted(e)) for e in _T.edges()))
_P = nx.spring_layout(_T2, seed=4, k=0.40, iterations=700)
_r = max(np.hypot(*v) for v in _P.values())
POS_MS = {n: (v[0] / _r * 2.05 * 0.92, v[1] / _r) for n, v in _P.items()}

def draw_musculoskeletal(g):
    # the links in order of the product of the shares of their ends, colored and widened by its square root
    pairs = sorted(_T.edges(), key=lambda e: P_MS[e[0]] * P_MS[e[1]])
    m = np.array([np.sqrt(P_MS[u] * P_MS[v]) for u, v in pairs]); lo, hi = m.min(), m.max()
    for (u, v), mm in zip(pairs, m):
        f = float((mm - lo) / (hi - lo + 1e-12))
        g.plot([POS_MS[u][0], POS_MS[v][0]], [POS_MS[u][1], POS_MS[v][1]], color=CMAP(CNORM(np.log2(mm * MS_N))),
               lw=0.30 + 2.0 * f, alpha=0.34 + 0.46 * f, zorder=1 + f, solid_capstyle='round')
    nn = sorted(_T.nodes(), key=lambda z: P_MS[z]); q = P_MS[nn]
    g.scatter([POS_MS[n][0] for n in nn], [POS_MS[n][1] for n in nn], s=size_of(q, 50.0, smin=2.0),
              c=CMAP(CNORM(np.log2(q * MS_N))), edgecolors=NODE_EDGE, linewidths=0.22, alpha=0.9, zorder=3,
              clip_on=False)
    g.set_xlim(-1.10 * 2.05 * 0.92, 1.10 * 2.05 * 0.92); g.set_ylim(-1.14, 1.14); g.axis('off')
    return POS_MS

STRUCT = {'cortex': dict(dH=CX_DH, ci=CX_CI, draw=draw_cortex, size=(0.72, 0.64), x=0.06, off=-0.02, bar=0.46,
                         label='cortex', note='in a motor task\n100 regions'),
          'muscles': dict(dH=E_['all'][0], ci=E_['all'][1], draw=draw_muscles, size=(0.74, 0.58), x=1.10,
                          off=-0.02, bar=0.52, label='muscles', note='in a stroke\n20 EMG sites'),
          'musculoskeletal': dict(dH=MS_DH, ci=None, draw=draw_musculoskeletal, size=(1.48, 0.76), x=1.92,
                                  off=-0.20, bar=0.64, label='musculoskeletal', note='on shared bones\n270 muscles')}
XL = (-0.50, 2.62)
YT, YB = (-0.11, 0.52), (0.72, 1.14)    # the two windows of the left plot, on one scale

def inset(fig, ax, x, y, size):
    W, H = fig.get_size_inches()
    fx, fy = fig.transFigure.inverted().transform(ax.transData.transform((x, y)))
    g = fig.add_axes([fx - size[0] / 2 / W, fy - size[1] / 2 / H, size[0] / W, size[1] / H])
    g.patch.set_alpha(0.0)
    return g

def extent(fig, g, ax, pos):
    # the lowest and the highest drawn element, in the data of the axes the drawing sits on
    xy = np.array(list(pos.values()), float)
    fig.canvas.draw_idle()
    d = g.transData.transform([[xy[:, 0].mean(), xy[:, 1].min()], [xy[:, 0].mean(), xy[:, 1].max()]])
    return ax.transData.inverted().transform(d)

def circles(fig, ax, x, y, ratio):
    # beginners open and experts filled, areas in the ratio of the effective numbers of landing cells
    W, H = fig.get_size_inches()
    fx, fy = fig.transFigure.inverted().transform(ax.transData.transform((x, y)))
    side = 0.24
    g = fig.add_axes([fx - side / 2 / W, fy - side / 2 / H, side / W, side / H])
    g.axis('off'); g.patch.set_alpha(0.0); g.set_xlim(-1.06, 1.06); g.set_ylim(-1.06, 1.06); g.set_aspect('equal')
    g.add_patch(Circle((0, 0), 1.0, facecolor='white', edgecolor=NEUTRAL, lw=0.8, zorder=1))
    g.add_patch(Circle((0, 0), ratio, facecolor=NEUTRAL, edgecolor=NEUTRAL, lw=0.5, zorder=2))

def draw_bits(fig, ox, oy, letter):
    W, H = fig.get_size_inches()
    l, b, w, h = BOX_L
    gap = 0.07
    k = (h - gap) / ((YT[1] - YT[0]) + (YB[1] - YB[0]))        # inches per bit, the same in both windows
    hb, ht = (YB[1] - YB[0]) * k, (YT[1] - YT[0]) * k
    axT = axes_at(fig, ox, oy, (l, b + hb + gap, w, ht)); axB = axes_at(fig, ox, oy, (l, b, w, hb))
    axR = axes_at(fig, ox, oy, BOX_R)
    vframe(axT, [0, 0.2, 0.4]); vframe(axB, [0.8, 1.0]); vframe(axR, [0, 1, 2], right=True)
    axT.set_ylim(YT[1], YT[0]); axB.set_ylim(YB[1], YB[0]); axR.set_ylim(2.12, -0.22)
    axT.spines['bottom'].set_visible(False); axB.spines['top'].set_visible(False)
    for ax in (axT, axB):
        ax.set_xlim(*XL); ax.set_xticks([]); ax.tick_params(axis='y', pad=2.0)
    for ax, yy, dy in ((axT, 0.0, 0.035 * hb / ht), (axB, 1.0, 0.035)):    # break marks where the windows meet
        for xx in (0.0, 1.0):
            ax.plot([xx - 0.010, xx + 0.010], [yy - dy, yy + dy], transform=ax.transAxes, color=INK, lw=0.6,
                    clip_on=False)
    axT.axhline(0.0, color=INK, lw=0.7, ls=(0, (3.2, 2.2)), zorder=1)        # the ceiling of the side panels
    ext = {}
    for key, s in STRUCT.items():
        ax = axB if s['dH'] > YT[1] else axT
        if s['ci'] is not None:
            ax.plot([s['x'] + s['bar']] * 2, s['ci'], color=NEUTRAL, lw=1.0, solid_capstyle='butt', zorder=3)
        ax.plot(s['x'] + s['bar'], s['dH'], ls='none', marker='_', ms=7.0, mew=1.4, color=INK, zorder=4)
        g = inset(fig, ax, s['x'] + s['off'], s['dH'], s['size'])
        ext[key] = extent(fig, g, ax, s['draw'](g))
        tx, _ = fig.transFigure.inverted().transform(axB.transData.transform((s['x'], 0)))
        fig.text(tx, (oy + b - 0.05) / H, s['label'], fontsize=6.2, color='k', ha='center', va='top')
    # each note reached by a dotted arrow from its drawing: under the two upper drawings, and above the
    # musculoskeletal drawing, over its label
    arrow = dict(arrowstyle='-|>', mutation_scale=5, lw=0.6, ls=(0, (1.2, 1.4)), color=NEUTRAL, shrinkA=3.0,
                 shrinkB=1.0)
    for key, gapb in (('cortex', 0.26), ('muscles', 0.25)):
        (xl, ylo), _ = ext[key]
        ny = STRUCT[key]['dH'] + gapb
        axT.text(xl, ny, STRUCT[key]['note'], fontsize=5.5, color='k', ha='center', va='top', linespacing=1.15)
        axT.annotate('', xy=(xl, ny - 0.004), xytext=(xl, ylo), arrowprops=arrow, annotation_clip=False)
    _, (xh, yhi) = ext['musculoskeletal']
    xm = STRUCT['musculoskeletal']['x']
    axT.text(xm, 0.455, STRUCT['musculoskeletal']['note'], fontsize=5.5, color='k', ha='center', va='bottom',
             linespacing=1.15)
    fig.add_artist(ConnectionPatch(xyA=(xh, yhi), coordsA=axB.transData, xyB=(xm, 0.462), coordsB=axT.transData,
                                   **arrow))
    # the outcome entropy beginners lose against experts, per stroke
    axR.set_xlim(-0.62, 1.70); axR.set_xticks([])
    axR.axhline(0.0, color=INK, lw=0.5, ls=(0, (1.0, 1.6)), zorder=1)
    for i, ty in enumerate(('clear', 'drive')):
        o = OUT[ty]
        axR.plot([i + 0.50] * 2, o['ci'], color=NEUTRAL, lw=1.0, solid_capstyle='butt', zorder=3)
        axR.plot(i + 0.50, o['obs'], ls='none', marker='_', ms=7.0, mew=1.4, color=INK, zorder=4)
        circles(fig, axR, i - 0.06, o['obs'], np.sqrt(2 ** o['exp'] / 2 ** o['beg']))
        tx, _ = fig.transFigure.inverted().transform(axR.transData.transform((i, 0)))
        fig.text(tx, (oy + b - 0.05) / H, ty, fontsize=6.2, color='k', ha='center', va='top')
    fig.text((ox + l - 0.30) / W, (oy + b + h / 2) / H, r'compression  $\Delta H$  (bits)', rotation=90,
             ha='center', va='center', fontsize=6.4)
    axR.set_ylabel(r'outcome  $\Delta h$  (bits)', labelpad=2.0)
    heading(fig, ox, oy, PH, letter, (BOX_L[0], BOX_R[0] + BOX_R[2]))

PANELS = [('A', PW_S, lambda f, x, y, L: draw_price(f, x, y, 'LAT', L)),
          ('B', PW_M, draw_bits),
          ('C', PW_S, lambda f, x, y, L: draw_price(f, x, y, 'BA', L))]

for letter, pw, draw in PANELS:
    fig = plt.figure(figsize=(pw, PH)); fig.set_label('Manuscript Fig 3' + letter)
    draw(fig, 0.0, 0.0, letter)
    plt.show()

for t in ('LAT', 'BA'):
    w = SHOW_W[t]; b = np.log2(w * len(w))
    print(f'{P2E.LABEL[KIND[t]]:<16} drawn realization: {len(w)} elements, H {-(w * np.log2(w)).sum():.4f} bits against the '
          f'ensemble mean {ENS[t]["H"].mean():.4f}; log2(p_i N) {b.min():+.2f} to {b.max():+.2f}')
CX_PAIRS = sorted({tuple(sorted((i, int(np.argmax(WCX[i]))))) for i in range(CX_N)})
_mir = {(int(a), int(b)) for a, b in zip(range(50), A17['partner'])}
CX_SAMENET = sum(A17['net'][u] == A17['net'][v] for u, v in CX_PAIRS)
CX_MIRROR = sum((u, v) in _mir for u, v in CX_PAIRS)
print(f'cortex drawn with {CX_N} regions and {len(CX_PAIRS)} links, the strongest coupling of every region: '
      f'{CX_SAMENET} join two regions of one network and {CX_MIRROR} join a region to its mirror image in the other '
      f'hemisphere; all {sum(RCX[u, v] > 0 for u, v in CX_PAIRS)} are positive correlations')
print(f'muscles drawn with 20 sites; musculoskeletal map drawn with {MS_N} muscles and {_T.number_of_edges()} links of '
      f'its {MS_L}')
print(f'outcome circles, expert radius over beginner radius: clear {np.sqrt(2 ** OUT["clear"]["exp"] / 2 ** OUT["clear"]["beg"]):.3f}, '
      f'drive {np.sqrt(2 ** OUT["drive"]["exp"] / 2 ** OUT["drive"]["beg"]):.3f}')

Regular lattice  drawn realization: 100 elements, H 6.6439 bits against the ensemble mean 6.6439; log2(p_i N) +0.00 to +0.00
Scale-free       drawn realization: 100 elements, H 6.3130 bits against the ensemble mean 6.3176; log2(p_i N) -2.41 to +1.96
cortex drawn with 100 regions and 73 links, the strongest coupling of every region: 53 join two regions of one network and 21 join a region to its mirror image in the other hemisphere; all 73 are positive correlations
muscles drawn with 20 sites; musculoskeletal map drawn with 270 muscles and 269 links of its 2679
outcome circles, expert radius over beginner radius: clear 0.582, drive 0.847


## Step 4. The figure set together

In [14]:
fig = plt.figure(figsize=(FIG_W, PH)); fig.set_label('Manuscript Fig 3')
x = 0.0
for letter, pw, draw in PANELS:
    draw(fig, x, 0.0, letter); x += pw
bold = sorted(t.get_text() for t in fig.findobj(mpl.text.Text) if t.get_fontweight() in ('bold', 700) and t.get_text().strip())
print('bold type only in the panel letters:', bold == list('ABC'))
plt.show()

bold type only in the panel letters: True


## Step 5. Every number the text takes from this notebook

In [15]:
VALS = {}
def put(key, v, fmt):
    s_ = format(v, fmt)
    VALS[key] = s_[1:] if s_.startswith('-') and float(s_.replace(',', '')) == 0 else s_   # no negative zero

put('F3_CEIL', CEIL, '.4f'); put('F3_LINKS', P2E.M_LINKS, 'd'); put('F3_NREAL', P2E.N_REAL, 'd')
for t in TOPOS:
    e = ENS[t]
    put(f'F3_H_{t}', e['H'].mean(), '.4f'); put(f'F3_DH_{t}', max(CEIL - e['H'].mean(), 0.0), '.3f')
    lo_, hi_ = P2E.CI[KIND[t]]                      # the interval of H of the second paper, turned into bits saved
    put(f'F3_DH_{t}_LO', max(CEIL - hi_, 0.0), '.3f'); put(f'F3_DH_{t}_HI', max(CEIL - lo_, 0.0), '.3f')
    put(f'F3_JC_{t}', max(CEIL - e['H'].mean(), 0.0) / CEIL, '.4f'); put(f'F3_DEFF_{t}', 2 ** e['H'].mean(), '.1f')
    put(f'F3_DHDEG_{t}', max(CEIL - e['Hdeg'].mean(), 0.0), '.3f'); put(f'F3_KAPPA_{t}', e['kappa'].mean(), '.2f')
put('F3_DH_WS_MIN', DH['WS'].min(), '.3f'); put('F3_DH_WS_MAX', DH['WS'].max(), '.3f')
w = SHOW_W['BA']; put('F3_DRAWN_BA_H', -(w * np.log2(w)).sum(), '.4f')
# cortex
put('F3_N_CX', CX_N, 'd'); put('F3_H_CX', CX_H, '.4f'); put('F3_DH_CX', CX_DH, '.3f')
put('F3_DH_CX_LO', CX_CI[0], '.3f'); put('F3_DH_CX_HI', CX_CI[1], '.3f'); put('F3_JC_CX', CX_DH / np.log2(CX_N), '.4f')
put('F3_ATLAS_RHO', A17['rho'], '.3f'); put('F3_ATLAS_NULL', A17['null'], '.3f'); put('F3_ATLAS_SD', A17['sd'], '.3f')
put('F3_ATLAS_RHO7', ATLAS['7-network']['rho'], '.3f'); put('F3_ATLAS_RHOK', ATLAS['Kong 2022']['rho'], '.3f')
put('F3_ATLAS_HOMO', A17['homo'], '.3f'); put('F3_ATLAS_ALL', RCX[iu].mean(), '.3f')
put('F3_ATLAS_SAME', A17['same'], '.3f'); put('F3_ATLAS_OTHER', A17['other'], '.3f')
put('F3_CX_LINKS', len(CX_PAIRS), 'd'); put('F3_CX_SAMENET', CX_SAMENET, 'd'); put('F3_CX_MIRROR', CX_MIRROR, 'd')
# musculoskeletal map
put('F3_N_MSK', MS_N, 'd'); put('F3_L_MSK', MS_L, ',d'); put('F3_H_MSK', MS_H, '.4f'); put('F3_DH_MSK', MS_DH, '.3f')
put('F3_JC_MSK', MS_DH / np.log2(MS_N), '.4f'); put('F3_DHDEG_MSK', MS_DHD, '.3f')
put('F3_BONES', B.shape[0], 'd'); put('F3_ATTACH', int(B.sum()), ',d')
put('F3_N_MSK10', M10_N, 'd'); put('F3_L_MSK10', M10_L, ',d'); put('F3_DH_MSK10', M10_DH, '.3f')
put('F3_MSK_TOP', P_MS.max() * MS_N, '.2f')
# muscles and joints of the stroke
put('F3_NREC', len(T), 'd'); put('F3_NPLAYER', T['sub'].nunique(), 'd'); put('F3_NSTROKE', int(T.n.sum()), ',d')
put('F3_NCLEAR', int(T[T.type == 'clear'].n.sum()), ',d'); put('F3_NDRIVE', int(T[T.type == 'drive'].n.sum()), ',d')
put('F3_NANNOT', len(ANN), ',d')
for ty in ('clear', 'drive'):
    for sk, tag in (('Beginner', 'B'), ('Intermediate', 'I'), ('Expert', 'E')):
        put(f'F3_N{tag}_{ty.upper()}', int(((T.type == ty) & (T.skill == sk)).sum()), 'd')
put('F3_RATE_LEG', RATE[0], '.0f'); put('F3_RATE_LOW', RATE[1], '.0f'); put('F3_RATE_UP', RATE[2], '.0f')
put('F3_RATE_JNT', RATE[3], '.0f')
for i, tag in enumerate(('LEG', 'LOW', 'UP', 'JNT')):
    put(f'F3_RATE_{tag}_MIN', RATES[:, i].min(), '.0f'); put(f'F3_RATE_{tag}_MAX', RATES[:, i].max(), '.0f')
for k, tag in (('dHE', 'EMG'), ('dHJ', 'JNT')):
    s = ST[k]
    put(f'F3_DH_{tag}', s['all'][0], '.3f'); put(f'F3_DH_{tag}_LO', s['all'][1][0], '.3f'); put(f'F3_DH_{tag}_HI', s['all'][1][1], '.3f')
    for sk, t3 in (('Beginner', 'BEG'), ('Intermediate', 'INT'), ('Expert', 'EXP')):
        put(f'F3_DH_{tag}_{t3}', s[sk][0], '.3f'); put(f'F3_DH_{tag}_{t3}_LO', s[sk][1][0], '.3f')
        put(f'F3_DH_{tag}_{t3}_HI', s[sk][1][1], '.3f')
    for ty in ('clear', 'drive'):
        put(f'F3_DH_{tag}_{ty.upper()}', s[ty][0], '.3f')
        put(f'F3_RHO_{tag}_{ty.upper()}', s['rho_' + ty][0], '+.2f'); put(f'F3_P_{tag}_{ty.upper()}', s['rho_' + ty][1], '.2f')
    put(f'F3_TASK_{tag}', s['task'][0], '+.3f'); put(f'F3_TASK_{tag}_LO', s['task'][1][0], '+.3f')
    put(f'F3_TASK_{tag}_HI', s['task'][1][1], '+.3f')
    put(f'F3_DH_{tag}_CLEAR_LO', s['clear'][1][0], '.3f'); put(f'F3_DH_{tag}_CLEAR_HI', s['clear'][1][1], '.3f')
    put(f'F3_DH_{tag}_DRIVE_LO', s['drive'][1][0], '.3f'); put(f'F3_DH_{tag}_DRIVE_HI', s['drive'][1][1], '.3f')
put('F3_CTRL_EMG', CTRL['E'][:, 0].mean(), '.3f'); put('F3_CTRL_JNT', CTRL['J'][:, 0].mean(), '.4f')
put('F3_CTRL_EMG_P95', CTRL['E'][:, 1].max(), '.3f'); put('F3_CTRL_JNT_P95', CTRL['J'][:, 1].max(), '.4f')
put('F3_DH_EMG_MIN', T.dHE.min(), '.3f'); put('F3_DH_JNT_MIN', T.dHJ.min(), '.4f')
put('F3_CEIL_EMG', np.log2(20), '.4f'); put('F3_CEIL_JNT', np.log2(21), '.4f'); put('F3_CEIL_MSK', np.log2(MS_N), '.4f')
put('F3_CEIL_CX', np.log2(CX_N), '.4f')
put('F3_H_EMG', np.log2(20) - ST['dHE']['all'][0], '.4f'); put('F3_H_JNT', np.log2(21) - ST['dHJ']['all'][0], '.4f')
put('F3_JC_EMG', ST['dHE']['all'][0] / np.log2(20), '.4f'); put('F3_JC_JNT', ST['dHJ']['all'][0] / np.log2(21), '.4f')
put('F3_DHDEG_EMG', T.dHEdeg.mean(), '.3f'); put('F3_DHDEG_JNT', T.dHJdeg.mean(), '.3f')
put('F3_H_MSK10', M10_H, '.4f'); put('F3_JC_MSK10', M10_DH / np.log2(M10_N), '.4f')
_, _, _, M10_DHD = comp(A10, 'deg'); put('F3_DHDEG_MSK10', M10_DHD, '.3f')
put('F3_EMG_PMIN', P_E.min() * 20, '.2f'); put('F3_EMG_PMAX', P_E.max() * 20, '.2f')
put('F3_EMG_IN_LOW', within[0], '.2f'); put('F3_EMG_IN_UP', within[1], '.2f'); put('F3_EMG_IN_LEG', within[2], '.2f')
put('F3_EMG_BETWEEN', between, '.2f')
# outcome
for ty in ('clear', 'drive'):
    o = OUT[ty]; T_ = ty.upper()
    put(f'F3_DHO_{T_}', o['obs'], '.2f'); put(f'F3_DHO_{T_}_LO', o['ci'][0], '.2f'); put(f'F3_DHO_{T_}_HI', o['ci'][1], '.2f')
    put(f'F3_DHO_{T_}_REACH', o['reach'], ',d'); put(f'F3_DHO_{T_}_P', o['reach'] / 10000, '.3f')
    put(f'F3_HO_{T_}_BEG', o['beg'], '.2f'); put(f'F3_HO_{T_}_INT', o['inter'], '.2f'); put(f'F3_HO_{T_}_EXP', o['exp'], '.2f')
    put(f'F3_DHO_{T_}_LAND', o['landed'], '.2f')
    put(f'F3_CELLS_{T_}_BEG', 2 ** o['beg'], '.1f'); put(f'F3_CELLS_{T_}_EXP', 2 ** o['exp'], '.1f')
for k_, v_ in VALS.items():
    print(f'{k_} = {v_}')
pub = {'F3_CEIL': '6.6439', 'F3_H_WS': '6.5832', 'F3_H_ER': '6.4820', 'F3_H_BA': '6.3176', 'F3_DH_WS': '0.061',
       'F3_DH_ER': '0.162', 'F3_DH_BA': '0.326', 'F3_DEFF_BA': '79.8', 'F3_KAPPA_ER': '6.90', 'F3_KAPPA_BA': '10.10',
       'F3_H_CX': '6.5468'}
bad = [f'{k} {VALS[k]} against {v}' for k, v in pub.items() if VALS[k] != v]
print()
print(len(VALS), 'values, every one finite:', all(np.isfinite(float(v.replace(',', ''))) for v in VALS.values()))
print('check, every value the second paper published is returned:', not bad, '; '.join(bad))

F3_CEIL = 6.6439
F3_LINKS = 300
F3_NREAL = 100
F3_H_LAT = 6.6439
F3_DH_LAT = 0.000
F3_DH_LAT_LO = 0.000
F3_DH_LAT_HI = 0.000
F3_JC_LAT = 0.0000
F3_DEFF_LAT = 100.0
F3_DHDEG_LAT = 0.000
F3_KAPPA_LAT = 6.00
F3_H_WS = 6.5832
F3_DH_WS = 0.061
F3_DH_WS_LO = 0.055
F3_DH_WS_HI = 0.067
F3_JC_WS = 0.0091
F3_DEFF_WS = 95.9
F3_DHDEG_WS = 0.011
F3_KAPPA_WS = 6.09
F3_H_ER = 6.4820
F3_DH_ER = 0.162
F3_DH_ER_LO = 0.157
F3_DH_ER_HI = 0.167
F3_JC_ER = 0.0244
F3_DEFF_ER = 89.4
F3_DHDEG_ER = 0.113
F3_KAPPA_ER = 6.90
F3_H_BA = 6.3176
F3_DH_BA = 0.326
F3_DH_BA_LO = 0.324
F3_DH_BA_HI = 0.329
F3_JC_BA = 0.0491
F3_DEFF_BA = 79.8
F3_DHDEG_BA = 0.357
F3_KAPPA_BA = 10.10
F3_DH_WS_MIN = 0.024
F3_DH_WS_MAX = 0.231
F3_DRAWN_BA_H = 6.3130
F3_N_CX = 100
F3_H_CX = 6.5468
F3_DH_CX = 0.097
F3_DH_CX_LO = 0.069
F3_DH_CX_HI = 0.126
F3_JC_CX = 0.0146
F3_ATLAS_RHO = -0.340
F3_ATLAS_NULL = 0.001
F3_ATLAS_SD = 0.015
F3_ATLAS_RHO7 = -0.191
F3_ATLAS_RHOK = -0.076
F3_ATLAS_HOMO = 0.451
F3_ATLAS_ALL = 0.020
F3_ATLAS_SAME = 0.359
F